<a href="https://colab.research.google.com/github/MoeTahech/real-fake-face-detection-thesis/blob/main/notebooks/07_third_domain_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 — Phase 7: Third-Domain Generalization Test

Phase 6 showed that training on two forgery types (StyleGAN + face-swap)
dramatically improved generalization to those two types. This phase asks the
natural follow-up question: **does that improvement extend to a completely
unseen third forgery type**, or did the model just memorize two specific
artifact styles?

We test on **diffusion-generated faces** (Stable Diffusion v1.4) — a
genuinely different generation method from both GANs (StyleGAN) and
autoencoder-based face-swapping, and one neither Phase 2 nor Phase 6 ever
saw during training.

Compares three models on this new, unseen domain:
- Phase 2 (single-source, StyleGAN only)
- Phase 6 (mixed-source, StyleGAN + face-swap)

Requires `02_finetuning.ipynb` and `06_mixed_source_training.ipynb` to have
been run at least once (checkpoints in Drive).

Runtime → Change runtime type → GPU, then run cells top to bottom.

### Step 1 — Confirm you have a GPU

In [ ]:
!nvidia-smi

Tue Aug 11 03:00:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/thesis_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be loaded from:", CHECKPOINT_DIR)

Mounted at /content/drive
Checkpoints will be loaded from: /content/drive/MyDrive/thesis_checkpoints


### Step 3 — Kaggle API access (via Colab Secrets)

Requires secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` already set up (🔑 icon,
left sidebar).

In [ ]:
import os

os.environ["KAGGLE_API_TOKEN"] = "KGAT_305e81603250999b12324cc02e529f89"

### Step 4 — Download the third-domain dataset (Stable Diffusion faces) and the StyleGAN dataset (for a matched real-image pool)

The Stable Diffusion dataset is fakes-only, so we pair it with real photos
from the StyleGAN dataset's held-out test split — real photos never used in
any training so far, and independent of forgery method.

In [ ]:
!kaggle datasets download -d bwandowando/faces-dataset-using-stable-diffusion-v14
!unzip -q faces-dataset-using-stable-diffusion-v14.zip -d data_diffusion
print("Diffusion dataset downloaded.")

!kaggle datasets download -d xhlulu/140k-real-and-fake-faces
!unzip -q 140k-real-and-fake-faces.zip -d data_raw
print("StyleGAN dataset downloaded (for matched real images).")

Dataset URL: https://www.kaggle.com/datasets/bwandowando/faces-dataset-using-stable-diffusion-v14
License(s): CC0-1.0
100% 1.10G/1.10G [00:15<00:00, 74.0MB/s]

Diffusion dataset downloaded.
Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:26<00:00, 150MB/s]

StyleGAN dataset downloaded (for matched real images).


### Step 5 — Locate all diffusion-generated images and the StyleGAN real test images

The diffusion dataset's internal folder layout isn't guaranteed, so we search
recursively for all image files rather than assuming a fixed structure.

In [ ]:
import os

def find_all_images(root, exts=(".jpg", ".jpeg", ".png")):
    found = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith(exts):
                found.append(os.path.join(dirpath, fname))
    return found

diffusion_images = find_all_images("data_diffusion")
print("Diffusion-generated images found:", len(diffusion_images))
print("Example paths:", diffusion_images[:3])

def find_real_fake_folders(root):
    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        lower = [d.lower() for d in dirnames]
        if "real" in lower and "fake" in lower:
            candidates.append(dirpath)
    return candidates

def find_split(root, keyword):
    candidates = find_real_fake_folders(root)
    matches = [x for x in candidates if keyword in x.lower()]
    return matches[0] if matches else (candidates[0] if candidates else None)

def find_class_dir(split_root, class_name):
    for d in os.listdir(split_root):
        if d.lower() == class_name:
            return os.path.join(split_root, d)
    raise FileNotFoundError(f"No '{class_name}' folder in {split_root}")

styleGAN_test = find_split("data_raw", "test")
real_dir = find_class_dir(styleGAN_test, "real")
real_images = [os.path.join(real_dir, f) for f in os.listdir(real_dir)]
print("StyleGAN test real images found:", len(real_images))

Diffusion-generated images found: 2536
Example paths: ['data_diffusion/Validation/Female/seed_263143_00021.png', 'data_diffusion/Validation/Female/seed_263219_00097.png', 'data_diffusion/Validation/Female/seed_263146_00024.png']
StyleGAN test real images found: 10000


### Step 6 — Build the third-domain evaluation set

Capped at 5,000 images per class (or fewer if the diffusion dataset has less)
for a manageable evaluation size, matching the scale used in earlier
cross-dataset evaluations.

In [ ]:
import random

def build_eval_set(dst_root, fake_paths, real_paths, n_per_class, seed=7):
    rng = random.Random(seed)
    fake_dst = os.path.join(dst_root, "fake")
    real_dst = os.path.join(dst_root, "real")
    os.makedirs(fake_dst, exist_ok=True)
    os.makedirs(real_dst, exist_ok=True)

    fake_sample = fake_paths[:]
    rng.shuffle(fake_sample)
    fake_sample = fake_sample[:n_per_class]

    real_sample = real_paths[:]
    rng.shuffle(real_sample)
    real_sample = real_sample[:n_per_class]

    for i, path in enumerate(fake_sample):
        dst = os.path.join(fake_dst, f"{i}_{os.path.basename(path)}")
        if not os.path.exists(dst):
            os.symlink(os.path.abspath(path), dst)
    for i, path in enumerate(real_sample):
        dst = os.path.join(real_dst, f"{i}_{os.path.basename(path)}")
        if not os.path.exists(dst):
            os.symlink(os.path.abspath(path), dst)

    return len(fake_sample), len(real_sample)

n_fake, n_real = build_eval_set("eval_diffusion", diffusion_images, real_images, n_per_class=5000)
print(f"Third-domain eval set built: {n_fake} fake (diffusion), {n_real} real")

Third-domain eval set built: 2536 fake (diffusion), 5000 real


### Step 7 — Evaluate Phase 2 and Phase 6 on this unseen third domain

Neither model has ever seen diffusion-generated faces during training. This
directly tests whether Phase 6's generalization gain was specific to the two
forgery types it trained on, or reflects something more broadly transferable.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), normalize])

eval_set = datasets.ImageFolder("eval_diffusion", transform=transform)
eval_loader = DataLoader(eval_set, batch_size=64, shuffle=False, num_workers=2)
print("On-disk class mapping:", eval_set.class_to_idx)


def load_resnet(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.load_state_dict(checkpoint["model_state"])
    model.to(device).eval()
    return model, checkpoint["class_to_idx"]


def evaluate(model, ckpt_class_to_idx):
    real_idx = ckpt_class_to_idx["real"]
    remap = {idx: ckpt_class_to_idx[name.lower()] for name, idx in eval_set.class_to_idx.items()}

    all_labels, all_probs, all_preds = [], [], []
    with torch.no_grad():
        for images, labels in eval_loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, real_idx].cpu().numpy()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend([remap[l.item()] for l in labels])

    acc = accuracy_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    cm = confusion_matrix(all_labels, all_preds)
    return acc, auc, cm


phase2_model, phase2_map = load_resnet(f"{CHECKPOINT_DIR}/best_model_finetuned.pth")
phase6_model, phase6_map = load_resnet(f"{CHECKPOINT_DIR}/best_model_mixed.pth")

print(f"\n{'Model':<30} {'Accuracy':<12} {'AUC':<10}")
for name, model, cmap in [("Phase 2 (single-source)", phase2_model, phase2_map),
                            ("Phase 6 (mixed-source)", phase6_model, phase6_map)]:
    acc, auc, cm = evaluate(model, cmap)
    print(f"{name:<30} {acc:<12.4f} {auc:<10.4f}")
    print(f"  Confusion matrix:\n{cm}\n")

On-disk class mapping: {'fake': 0, 'real': 1}

Model                          Accuracy     AUC       
Phase 2 (single-source)        0.6545       0.2768    
  Confusion matrix:
[[   4 2532]
 [  72 4928]]

Phase 6 (mixed-source)         0.6636       0.5524    
  Confusion matrix:
[[ 370 2166]
 [ 369 4631]]

